In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib as plt
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np

import warnings
warnings.filterwarnings('ignore')
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score,roc_auc_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import roc_curve, auc
from xgboost import XGBClassifier



In [ ]:
df = pd.read_csv("Failures_Boosted.csv")

In [ ]:
df.head()

In [ ]:
df = df.drop(columns=['Unnamed: 0'])

In [ ]:
for i in (('L', 0), ('M', 1), ('H', 2)):
    df['Type'] = df['Type'].replace(*i)

In [ ]:
df.count()

In [ ]:
df = df.rename(columns={'Air temperature [K]':'Air_temperature','Process temperature [K]':'Process_temperature',
                        'Rotational speed [rpm]': 'Rotational_speed','Torque [Nm]':'Torque','Tool wear [min]':'Tool_wear','Time/Part':'Part_Time'})

In [ ]:
df

In [ ]:
num_data = ['Air_temperature','Process_temperature','Rotational_speed','Torque','Tool_wear','Temp_Delta', 'Power']
categorical = ['Type', 'Part_Time']

target1 = ['Failures']
target2 = ['TWF']

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[num_data] = scaler.fit_transform(df[num_data])

In [ ]:
X = df.drop(columns=['Failures'])
y= df['Failures']

In [ ]:
y.head()

In [ ]:
X

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:

from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

class XGBoostTuner:
    def __init__(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train
        self.model = XGBClassifier()
        self.params = {
            'n_estimators': [1000, 1500, 2000],
            'max_depth': [2, 3, 6],
            'subsample': [0.90, 0.95, 1],
            'learning_rate': [0.01,0.05,0.1],
            'objective': ['binary:logistic', 'binary:hinge'],
            'min_child_weight': [5, 10, 15], 
            'reg_alpha':[0.1], 
            'reg_lambda':[0.1]
        }

    def tune_hyperparameters(self):     
        search = GridSearchCV(self.model, self.params, cv=5, scoring='roc_auc')
        search.fit(self.X_train, self.y_train)
        return search


xgboost_tuner = XGBoostTuner(X_train, y_train)
search_results = xgboost_tuner.tune_hyperparameters()

best_model = search_results.best_estimator_
score = search_results.best_score_

print("Model Training:")
print(best_model)

print("Results:")
print(score)



In [ ]:
import pickle
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

class XGBoostTuner:
    def __init__(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train
        self.model = XGBClassifier()
        self.params = {
            'n_estimators': [2000],
            'max_depth': [3],
            'subsample': [0.8],
            'learning_rate': [0.05],
            'objective': ['binary:logistic', 'binary:hinge'],
            'min_child_weight': [5], 
            'reg_alpha':[0.1], 
            'reg_lambda':[0.1]
        }

    def tune_hyperparameters(self):     
        search = GridSearchCV(self.model, self.params, cv=5, scoring='roc_auc')
        search.fit(self.X_train, self.y_train)
        return search


xgboost_tuner = XGBoostTuner(X_train, y_train)
search_results = xgboost_tuner.tune_hyperparameters()

best_model = search_results.best_estimator_
score = search_results.best_score_

with open('xgboost_model.pkl', 'wb') as file:
    pickle.dump(best_model, file)

print("Model Training:")
print(best_model)

print("Results:")
print(score)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

class Xgboost_test:
    def __init__(self, X_test, y_test, best_model):
        self.X_test = X_test
        self.y_test = y_test
        self.best_model = best_model 

    def evaluate_model(self):
        y_pred = self.best_model.predict(self.X_test)
        return y_pred

xgboost_tester = Xgboost_test(X_test, y_test, best_model)
y_pred_test = xgboost_tester.evaluate_model()

print('Classification Report (Test Data):')
print(classification_report(y_test, y_pred_test, digits=4))

print('Confusion Matrix (Test Data):')
cm_test = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm_test, annot=True, fmt='g', cmap='Blues')
plt.ylabel('Actual', fontsize=10)
plt.xlabel('Predicted', fontsize=10)
plt.title('Confusion Matrix (Test Data)', fontsize=12)
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_test)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = {:.2f})'.format(roc_auc))
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

In [ ]:
import pickle

with open('xgboost_tuner_model.pkl', 'wb') as file:
  pickle.dump(best_model, file)

#with open('xgboost_tuner_model.pkl', 'rb') as file:
#   loaded_model = pickle.load(file)